# Loading script for Green stay hotels in DWHPFX schema

In [24]:
import requests
import json
import pandas as pd
import time
import configparser
import pyexasol

In [35]:
def get_token(urlLogin, env):
    """ This function authenticates and returns token and refreshToken """
    auth = {"client_id": "f6eeddfe-b600-4fc8-b283-e66f5fa0e0aa", "client_secret": "45f8160f-0bee-43af-a7ac-3d1013c84a78"}
    response = requests.post(urlLogin, json=auth)
    if response.status_code == 200:
        data = response.text
        parsed = json.loads(data)
        token = parsed['token']
        refreshToken = parsed['refreshToken']
    return token, refreshToken

def GreenStayExtract(urlAuth, env='det'):
    """ This function fetches the data from the api and stores in a df """

    token, refreshToken = get_token(urlAuth, env)
    startTime = time.time()
    urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page=1&size=500"
    getResponse = requests.get(urlGet)
    rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                      'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                      'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                      'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                      'status': 'GREEN_INSPECTION_STATUS'
                      }

    if getResponse.status_code == 200:
        data = getResponse.text
        parsed = json.loads(data)
        print(f"Status code: {getResponse.status_code}")
    else:
        print("Error in fetching the pages")
        
    try:
        temp = []
        for x in range(int(parsed['total_pages'])):
            urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page={x+1}&size=500"
            getResponse = requests.get(urlIter)

            if getResponse.status_code == 200:
                data = getResponse.text
                parsed = json.loads(data)
                temp.append(parsed.get('results'))
            else:
                print("Request failed page: {} ".format(x))
        flatten = [item for sublist in temp for item in sublist]
        dfRaw = pd.DataFrame(flatten) 
        dfRaw['created_date'] = dfRaw['created_date'].astype(str).str[:-6]
        dfRaw['updated_date'] = dfRaw['updated_date'].astype(str).str[:-6]
        dfRaw['created_date'] = pd.to_datetime(dfRaw['created_date'], utc=False)
        dfRaw['updated_date'] = pd.to_datetime(dfRaw['updated_date'], utc=False)
        dfRaw = dfRaw[dfRaw.hkey.notnull()]
        dfRaw = dfRaw[['hkey', 'created_date', 'updated_date', 'report_year', 'kilogramCarbonPOC', 'literWaterPOC',
                       'kilogramWastePOC', 'carbonClass', 'waterClass', 'wasteClass', 'greenClass', 'type', 'status']]
        dfRaw.rename(columns=rename_columns, inplace=True)
        df = dfRaw.sort_values('UPDATED_DATE').groupby('HOTEL_ID').tail(1)
        df['CREATED_DATE'] = pd.to_datetime(dfRaw['CREATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        df['UPDATED_DATE'] = pd.to_datetime(dfRaw['UPDATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
        df['GREEN_INSPECTION_TYPE'] = df['GREEN_INSPECTION_TYPE'].apply(lambda x: x.upper())
        df['LDTS'] = time.strftime('%Y-%m-%d')
        print(f'Number of records having HOTEL_IDs {dfRaw.HOTEL_ID.nunique()}.')
        print(f'Number of records with duplicate HOTEL_IDs: {dfRaw.duplicated(subset="HOTEL_ID", keep="first").sum()}.')
        endTime = time.time() - startTime
        print(f'Time in minutes: {round(endTime / 60, 2)}')
    except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function GreenScore - ')
        print(f'Time in minutes: {round(endTime / 60, 2)}')
        raise e
    return df


def GreenStayLoad(df): 
    """ This function is to load the data into Exasol DB -> DWHPFX schema """
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\svi02\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.GREEN_STAY_HOTELS")
        connect.import_from_pandas(df, table = ('DWHPFX','GREEN_STAY_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except Exception as e:
        print('Failed in function GreenStayLoad - ')
        raise e

### Function call

In [36]:
df=GreenStayExtract('https://api.hotel-audit.hrs.com/auth/login')
# GreenStayLoad(df)
df.head()

Status code: 200


,0
0,89
1,103
2,110
3,206
4,217


### Test results in excel

In [106]:
from datetime import date
df.to_excel('C:\\Users\\svi02\\Documents\\misc\\'+str(date.today())+'_test.xlsx', 
              sheet_name='Sheet1', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )

In [14]:
import pandas as pd

data = [(-1, -1, 0), (-1, 1, 1), (1, -1, 1), (1, 1, 0)]

dftest = pd.DataFrame(data, columns=['feature1', 'feature2', 'label'])

C:\Anaconda\lib\site-packages\numpy\_distributor_init.py:32: UserWarning: loaded more than 1 DLL from .libs:
C:\Anaconda\lib\site-packages\numpy\.libs\libopenblas.QVLO2T66WEPI7JZ63PS3HMOHFEY472BC.gfortran-win_amd64.dll
C:\Anaconda\lib\site-packages\numpy\.libs\libopenblas.TXA6YQSD3GCQQC22GEQ54J2UDCXDXHWN.gfortran-win_amd64.dll
  stacklevel=1)


In [15]:
dftest.corr()

,feature1,feature2,label
feature1,1.0,0.0,0.0
feature2,0.0,1.0,0.0
label,0.0,0.0,1.0


In [11]:
lastname: str = 100

In [12]:
lastname

100

In [3]:
int = 'Lakshmi'

In [4]:
int

'Lakshmi'

In [5]:
lastname :int

In [6]:
lastname

'Lakshmi'

In [105]:
urlAuth='https://api.hotel-audit.hrs.com/auth/login'
token, refreshToken = get_token(urlAuth, env='det')
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report?token={token}&page=1&size=1"
getResponse = requests.get(urlGet)
rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                  'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                  'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                  'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                  'status': 'GREEN_INSPECTION_STATUS'
                  }

if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
    print(parsed['total_pages'])
    print(f"Status code: {getResponse.status_code}")
else:
    print("Error in fetching the pages")
# try:
#     temp=[]
#     for x in range(int(parsed['total_pages'])):
#         urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/green?token={token}&page={x+1}&size=500"
#         getResponse = requests.get(urlIter)

#         if getResponse.status_code == 200:
#             data = getResponse.text
#             parsed = json.loads(data)
#             temp.append(parsed.get('results').values())
#         else:
#             print("Request failed page: {} ".format(x))
#         flatten = [item for sublist in temp for item in sublist]
#         dfRaw = pd.DataFrame(flatten)
#         dfRaw['created_date'] = dfRaw['created_date'].astype(str).str[:-6]
#         dfRaw['updated_date'] = dfRaw['updated_date'].astype(str).str[:-6]
#         dfRaw['created_date'] = pd.to_datetime(dfRaw['created_date'], utc=False)
#         dfRaw['updated_date'] = pd.to_datetime(dfRaw['updated_date'], utc=False)
#         dfRaw = dfRaw[dfRaw.hkey.notnull()]
#         dfRaw = dfRaw[['hkey', 'created_date', 'updated_date', 'report_year', 'kilogramCarbonPOC', 'literWaterPOC',
#                        'kilogramWastePOC', 'carbonClass', 'waterClass', 'wasteClass', 'greenClass', 'type', 'status']]
#         dfRaw.rename(columns=rename_columns, inplace=True)
#         df = dfRaw.sort_values('UPDATED_DATE').groupby('HOTEL_ID').tail(1)
#         df['CREATED_DATE'] = pd.to_datetime(dfRaw['CREATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
#         df['UPDATED_DATE'] = pd.to_datetime(dfRaw['UPDATED_DATE'], format='%Y%m%d', errors='ignore').dt.date
#         df['GREEN_INSPECTION_TYPE'] = df['GREEN_INSPECTION_TYPE'].apply(lambda x: x.upper())
#         df['LDTS'] = time.strftime('%Y-%m-%d')
#         print(f'Number of records having HOTEL_IDs {dfRaw.HOTEL_ID.nunique()}.')
#         print(f'Number of records with duplicate HOTEL_IDs: {dfRaw.duplicated(subset="HOTEL_ID", keep="first").sum()}.')
#         endTime = time.time() - startTime
#         print(f'Time in minutes: {round(endTime / 60, 2)}')
# except Exception as e:
#         endTime = time.time() - startTime
#         print('Failed in function GreenScore - ')
#         print(f'Time in minutes: {round(endTime / 60, 2)}')
#         raise e

67549
Status code: 200


In [106]:
parsed

{'results': [{'id': '00005b15-b99c-444d-97bf-d3cd02f8c8db',
   'hkey': 431251,
   'name': 'Homewood Suites by Hilton Lawrenceville',
   'created_date': '2020-07-31T16:16:24.000Z',
   'updated_date': '2020-07-31T16:16:24.000Z',
   'checked': '1,2,3,4,5,6,8,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46',
   'link': 'https://www.hilton.com/en/corporate/cleanstay/',
   'program_name': 'Hilton CleanStayTM',
   'version': 1,
   'terms': 2,
   'services': None,
   'audit_date': None,
   'auditor_key': None,
   'chain_id': 240,
   'chain_name': 'Hilton Hotels Corporation',
   'brand_id': 1537,
   'brand_name': 'Homewood Suites by Hilton',
   'city_id': 168172,
   'city': 'Lawrenceville (Georgia)',
   'country_id': 165,
   'country': 'USA',
   'performance_cluster': None,
   'status': True,
   'type': 'cleansafe_self_inspection',
   'missed': ''}],
 'page_number': 1,
 'page_size': 1,
 'total_pages': 67549}

In [55]:
urlAuth='https://api.hotel-audit.hrs.com/auth/login'
token, refreshToken = get_token(urlAuth, env='det')
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page=1&size=500"
getResponse = requests.get(urlGet)
rename_columns = {'hkey': 'HOTEL_ID', 'created_date': 'CREATED_DATE', 'updated_date': 'UPDATED_DATE',
                  'report_year': 'REPORTING_YEAR', 'kilogramCarbonPOC': 'CARBON_KGS', 'literWaterPOC': 'WATER_LTS',
                  'kilogramWastePOC': 'WASTE_KGS', 'carbonClass': 'CARBON_CLASS', 'waterClass': 'WATER_CLASS',
                  'wasteClass': 'WASTE_CLASS', 'greenClass': 'GREEN_CLASS', 'type': 'GREEN_INSPECTION_TYPE',
                  'status': 'GREEN_INSPECTION_STATUS'
                  }

if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
#     print(parsed)
    print(f"Status code: {getResponse.status_code}")
else:
    print("Error in fetching the pages")
try:
    dfRaw = pd.DataFrame()
    temp1=[]
    for x in range(2):
        urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page={x+1}&size=500"
        getResponse = requests.get(urlIter)

        if getResponse.status_code == 200:
            data = getResponse.text
            parsed = json.loads(data)
            temp1.append(parsed.get('results'))
        else:
            print("Request failed page: {} ".format(x))
except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function GreenScore - ')
        print(f'Time in minutes: {round(endTime / 60, 2)}')
        raise e
temp1

Status code: 200


[[{'hkey': -1,
   'gs_id': 'Q930301291',
   'gs_distance': 0.10350994103402734,
   'date': '2021-09-02T10:00:07.000Z',
   'gs_province': '',
   'gs_region': 'Africa',
   'gs_district': 'Beach Road',
   'gs_city': 'Sekondi - Takoradi',
   'gs_country': 'Ghana',
   'gs_type': 'None',
   'gs_countrycode': 'GH',
   'gs_population': 0,
   'composite': 56,
   'nightime': 49,
   'physical': 55,
   'women': 51,
   'theft': 61,
   'freedom': 68,
   'health': 52,
   'lgbtq': 51,
   'type': 'geosure',
   'status': True},
  {'hkey': 1,
   'gs_id': 'Q930406986',
   'gs_distance': 0.0002792566589836676,
   'date': '2021-09-02T10:00:07.000Z',
   'gs_province': 'Berlin',
   'gs_region': 'European Union',
   'gs_district': 'Tiergarten Süd',
   'gs_city': 'Berlin',
   'gs_country': 'Germany',
   'gs_type': 'Capital',
   'gs_countrycode': 'DE',
   'gs_population': 0,
   'composite': 69,
   'nightime': 64,
   'physical': 70,
   'women': 68,
   'theft': 45,
   'freedom': 83,
   'health': 72,
   'lgbtq': 76

In [67]:
temp1[1][0]

{'hkey': 553,
 'gs_id': 'Q930205327',
 'gs_distance': 0.0005261024484226414,
 'date': '2021-09-02T10:00:07.000Z',
 'gs_province': 'Florida',
 'gs_region': 'North America',
 'gs_district': 'Northside',
 'gs_city': 'Jacksonville',
 'gs_country': 'United States',
 'gs_type': 'None',
 'gs_countrycode': 'US',
 'gs_population': 0,
 'composite': 68,
 'nightime': 63,
 'physical': 66,
 'women': 63,
 'theft': 65,
 'freedom': 79,
 'health': 62,
 'lgbtq': 73,
 'type': 'geosure',
 'status': True}